
# Sélection aléatoire de maladies — GBD niveau 3 (5 catégories × 5 maladies)

**Objectif** : construire un échantillon de maladies **non biaisé par notre hypothèse de départ**
(féminisme / santé), en suivant une règle indépendante et reproductible :

1. Partir des maladies de **niveau 3** du *Global Burden of Disease Study* (GBD).
2. Regrouper ces maladies sous 5 **catégories parentes de niveau 2** du GBD.
3. Tirer aléatoirement **5 maladies par catégorie** (`random_state=42` pour la reproductibilité).
4. Ce n'est **qu'ensuite** qu'on ajoutera les variables liées à l'hypothèse (proportion de femmes
   atteintes, maladies spécifiques aux femmes, nombre de publications, financement, délai
   diagnostique...).

Les 5 catégories retenues (niveau 2 GBD) :
- Neurological disorders
- Cardiovascular diseases
- Neoplasms (cancers)
- Endocrine, metabolic, blood and immune disorders
- Musculoskeletal disorders

> ⚠️ **Note méthodologique importante** : le catalogue officiel complet des causes GBD (avec les
> `cause_id` exacts) est distribué par l'IHME via le **GBD Results Tool / GHDx**
> (https://ghdx.healthdata.org/gbd-results-tool), qui nécessite une inscription et n'est pas
> accessible par une simple requête HTTP anonyme. Ce notebook contient donc, par défaut, une
> **liste curatée "à la main"** des principales maladies de niveau 3 pour ces 5 catégories,
> construite à partir de la hiérarchie des causes GBD publiée dans la littérature (GBD 2019/2021
> cause list). Elle est correcte dans ses grandes lignes mais **pas garantie exhaustive ni
> parfaitement à jour**.
>
> Si tu as accès au fichier officiel des causes GBD (export CSV du GHDx, colonnes du type
> `cause_id`, `cause_name`, `level`, `parent_name`), **remplace directement la cellule "Étape 1"**
> par un `pd.read_csv("mon_fichier_causes_gbd.csv")` : tout le reste du notebook (filtrage,
> catégories, tirage aléatoire) fonctionnera à l'identique sans rien changer.


## Imports

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 200) # Set the maximum number of rows to display
pd.set_option("display.width", 120) # Set the width of the display


## Étape 1 — Liste des maladies GBD niveau 3

### Option A (par défaut) — liste curatée intégrée au notebook

Si tu n'as pas de fichier officiel, on part de la liste ci-dessous.

### Option B — charger le fichier officiel GBD (recommandé si disponible)

Décommente et adapte le chemin si tu as un export du GHDx / GBD Results Tool :

```python
# diseases_raw = pd.read_csv("causes_gbd.csv")
# Colonnes attendues au minimum : cause_id, cause_name, level, parent_name
```


In [9]:

# --- Option A : liste curatée de maladies GBD niveau 3, par catégorie parente (niveau 2) ---
# Sources : hiérarchie des causes GBD 2019/2021 (structure publique, cf. publications GBD /
# documentation IHME). Liste non exhaustive, à raffiner si un export officiel est disponible.

raw_data = {
    "Neurological disorders": [
        "Alzheimer's disease and other dementias",
        "Parkinson's disease",
        "Epilepsy",
        "Multiple sclerosis",
        "Motor neuron disease",
        "Migraine",
        "Tension-type headache",
        "Guillain-Barré syndrome",
        "Encephalitis",
        "Idiopathic developmental intellectual disability",
        "Headache disorders",
        "Tension-type headache",
    ],
    "Cardiovascular diseases": [
        "Ischemic heart disease",
        "Ischemic stroke",
        "Intracerebral hemorrhage",
        "Subarachnoid hemorrhage",
        "Hypertensive heart disease",
        "Cardiomyopathy and myocarditis",
        "Atrial fibrillation and flutter",
        "Rheumatic heart disease",
        "Aortic aneurysm",
        "Peripheral artery disease",
        "Endocarditis",
        "Non-rheumatic valvular heart disease",
    ],
    "Neoplasms": [
        "Breast cancer",
        "Prostate cancer",
        "Colon and rectum cancer",
        "Tracheal, bronchus, and lung cancer",
        "Ovarian cancer",
        "Cervical cancer",
        "Uterine cancer",
        "Liver cancer",
        "Stomach cancer",
        "Esophageal cancer",
        "Pancreatic cancer",
        "Non-Hodgkin lymphoma",
        "Leukemia",
        "Brain cancer",
        "Bladder cancer",
        "Kidney cancer",
        "Thyroid cancer",
        "Malignant skin melanoma",
        "Larynx cancer",
        "Multiple myeloma",
    ],
    "Endocrine, metabolic, blood and immune disorders": [
        "Diabetes mellitus type 1",
        "Diabetes mellitus type 2",
        "Gout",
        "Thyroid disorders",
        "Systemic lupus erythematosus",
        "Sickle cell disorders",
        "Thalassemias",
        "G6PD deficiency",
        "Addison's disease",
        "Other endocrine, metabolic, blood, and immune disorders",
    ],
    "Musculoskeletal disorders": [
        "Rheumatoid arthritis",
        "Osteoarthritis",
        "Low back pain",
        "Neck pain",
        "Gout (musculoskeletal manifestations)",
        "Juvenile idiopathic arthritis",
        "Fibromyalgia",
        "Other musculoskeletal disorders",
    ],
    "Mental disorders": [
        "Depressive disorders",
        "Anxiety disorders",
        "Bipolar disorder",
        "Schizophrenia",
        "Eating disorders",
        "Autism spectrum disorders",
        "Attention-deficit/hyperactivity disorder",
        "Conduct disorder",
        "Post-traumatic stress disorder",
        "Obsessive-compulsive disorder",
    ],
    "Chronic respiratory diseases": [
        "Chronic obstructive pulmonary disease",
        "Asthma",
        "Interstitial lung disease",
        "Pulmonary sarcoidosis"
        
    ],
    "Digestive diseases": [
        "Cirrhosis and other chronic liver diseases",
        "Gastritis and duodenitis",
        "Peptic ulcer disease",
        "Inflammatory bowel disease",
        "Pancreatitis",
        "Gallbladder and biliary diseases",
        "Appendicitis",
        "Vascular intestinal disorders",
        "Irritable bowel syndrome",
    ],
    "Skin and subcutaneous diseases": [
        "Dermatitis",
        "Psoriasis",
        "Acne vulgaris",
        "Alopecia areata",
        "Urticaria",
        "Scabies",
        "Fungal skin diseases",
        "Decubitus ulcer",
        "Other skin and subcutaneous diseases",
    ],
    "Sense organ diseases": [
        "Age-related and other hearing loss",
        "Cataract",
        "Glaucoma",
        "Age-related macular degeneration",
        "Refraction disorders",
        "Other vision loss",
    ],
    "Urinary, gynecological and reproductive diseases": [
        "Urinary tract infections",
        "Urolithiasis",
        "Benign prostatic hyperplasia",
        "Male infertility",
        "Interstitial nephritis",
        "Endometriosis",
        "Uterine fibroids",
        "Genital prolapse",
        "Premenstrual syndrome",
        "Female infertility",
        "Polycystic ovary syndrome",
    ],
}

rows = []
for category, diseases_list in raw_data.items():
    for name in diseases_list:
        rows.append({
            "cause_name": name,
            "parent": category,
        })

diseases = pd.DataFrame(rows)
diseases

,cause_name,parent
0,Alzheimer's disease and other dementias,Neurological disorders
1,Parkinson's disease,Neurological disorders
2,Epilepsy,Neurological disorders
3,Multiple sclerosis,Neurological disorders
4,Motor neuron disease,Neurological disorders
5,Migraine,Neurological disorders
6,Tension-type headache,Neurological disorders
7,Guillain-Barré syndrome,Neurological disorders
8,Encephalitis,Neurological disorders
9,Idiopathic developmental intellectual disability,Neurological disorders



## Étape 2 — Définir nos 6 catégories

On restreint le tableau aux 6 grandes catégories parentes (niveau 2) retenues.


## Étape 5 — Enrichissement automatique : nombre de publications scientifiques

Parmi les variables listées plus haut, le **nombre de publications** est la seule qu'on peut
récupérer entièrement automatiquement, via deux API gratuites et sans clé :

- **PubMed / NCBI E-utilities** : `esearch.fcgi` renvoie un simple compte de résultats pour une
  requête donnée.
- **OpenAlex** : API ouverte, plus large que PubMed (toutes disciplines, pas seulement biomédical),
  utile en comparaison / vérification croisée.

Les autres variables (proportion de femmes atteintes, financement, délai diagnostique) ne sont
**pas** automatisables de façon fiable — elles nécessitent une recherche ciblée par maladie. On
crée donc des colonnes vides pour elles, à remplir manuellement (voir sources suggérées dans la
dernière cellule markdown).

> ⚠️ **Note d'environnement** : si tu exécutes ce notebook dans un environnement au réseau
> restreint (ex. un sandbox sans accès sortant libre), les appels à `eutils.ncbi.nlm.nih.gov` et
> `api.openalex.org` peuvent échouer (erreur réseau/403). Le code gère ce cas proprement (il met
> `NaN` et continue) mais **pour obtenir les vrais chiffres, exécute ce notebook dans un
> environnement avec accès internet normal** (Jupyter local, Google Colab, VS Code, etc.).

In [11]:
import numpy as np
import pandas as pd
import requests
import time

def count_pubmed(query: str, retries: int = 2, timeout: int = 10) -> float:
    """Retourne le nombre de résultats PubMed pour une requête, ou NaN en cas d'échec."""
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {"db": "pubmed", "term": query, "retmode": "json"}
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return int(r.json()["esearchresult"]["count"])
        except Exception:
            time.sleep(1)
    return np.nan

pubmed_counts = []
for disease in diseases["cause_name"]:
    pubmed_counts.append(count_pubmed(disease))
    time.sleep(0.34)  # reste sous la limite de ~3 requêtes/seconde sans clé API NCBI

diseases["publications_pubmed"] = pubmed_counts

n_echecs = sum(1 for count in pubmed_counts if pd.isna(count))
if n_echecs == len(diseases):
    print("⚠️ Toutes les requêtes ont échoué : le réseau semble bloqué dans cet environnement. "
          "Relance ce notebook depuis un environnement avec accès internet normal.")
else:
    print(f"Publications récupérées pour {len(diseases) - n_echecs}/{len(diseases)} maladies.")

pubmed_df = diseases[["cause_name", "parent", "publications_pubmed"]] \
    .sort_values("publications_pubmed", ascending=False)

pubmed_df

Publications récupérées pour 111/111 maladies.


,cause_name,parent,publications_pubmed
12,Ischemic heart disease,Cardiovascular diseases,618068
24,Breast cancer,Neoplasms,586689
80,Pancreatitis,Digestive diseases,466175
36,Leukemia,Neoplasms,402993
31,Liver cancer,Neoplasms,377283
37,Brain cancer,Neoplasms,292653
73,Asthma,Chronic respiratory diseases,243961
25,Prostate cancer,Neoplasms,241109
45,Diabetes mellitus type 2,"Endocrine, metabolic, blood and immune disorders",233095
2,Epilepsy,Neurological disorders,205488



## Étape 6 — Colonnes à compléter manuellement

Ces variables n'ont pas de source unique automatisable ; on prépare les colonnes vides, à remplir
maladie par maladie à partir des sources suivantes :

| Variable | Où chercher |
|---|---|
| `proportion_femmes_pct` | GBD Sex Differences tool, GBD Results Tool (filtre sexe), WHO Global Health Observatory |
| `specifique_femmes` (oui/non) | jugement à partir de la nature de la maladie (ex. cancer de l'ovaire = oui) |
| `financement_recherche` | NIH RePORTER (US), Wellcome Open Data / UK HRCS (UK) |
| `delai_diagnostique_moyen` | revue de littérature ciblée (PubMed/OpenAlex, requêtes du type `"[maladie] diagnostic delay"`) |
| `source_delai` | référence de l'étude utilisée pour `delai_diagnostique_moyen` |
